# 01 - Explore the data and run LaBSE curation

Loads the training and test data for one language, shows basic statistics, then embeds every training transcript with LaBSE (contextual sentence embeddings) to:

- cap near-identical sentences (cosine >= 0.90) at 3 recordings each;
- drop the 2% of transcripts furthest from the language centroid (atypical / noisy text).

The kept IDs are saved to `data/curation/{lang}_labse.json` and used by `scripts/train_lora.py --arm labse`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

import matplotlib.pyplot as plt
import pandas as pd

from echo.curation import run_curation
from echo.data import load_language_splits

In [ ]:
LANG = "sq"  # "sq" (Albanian) or "mk" (Macedonian)

In [ ]:
splits = load_language_splits(LANG)
train_df = splits["train"].select_columns(["uid", "source", "text"]).to_pandas()
test_df = splits["test"].select_columns(["uid", "source", "text"]).to_pandas()

pd.concat({"train": train_df.source.value_counts(), "test": test_df.source.value_counts()}, axis=1).fillna(0).astype(int)

In [ ]:
train_df["n_words"] = train_df.text.str.split().str.len()

plt.figure(figsize=(8, 4))
plt.hist(train_df.n_words, bins=50, color="#808080")
plt.xlabel("Words per transcript")
plt.ylabel("Count")
plt.title(f"Transcript length ({LANG}, train)")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

print("Unique transcripts:", train_df.text.nunique(), "of", len(train_df))

## LaBSE curation

In [ ]:
result = run_curation(LANG, splits["train"])
pd.Series(result.stats)

In [ ]:
texts = train_df.set_index("uid")["text"]

print("Near-duplicates removed (sample):")
for uid in result.near_duplicate_uids[:10]:
    print(" -", texts[uid])

print("\nOutliers removed (sample):")
for uid in result.outlier_uids[:10]:
    print(" -", texts[uid])

Re-run the notebook with `LANG = "mk"` to curate the Macedonian data.